# Transformer

In [1]:
import torch

## Basic Modules

- Positional Encoding
- FNN
- MHA


**Positional Encoding**

$$
PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{\text{model}}}) \\
PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})
$$

Note the division term $\frac{1}{10000^{\frac{2i}{d}}} = 10000^{-\frac{2i}{d}}$ is usually implimented to avoid using `pow()` for efficiency improvement. Because $a^b = e^{b\ln a}$, thus

$$10000^{-\frac{2i}{d}} = e^ {-\frac{2i}{d}\ln 10000}$$

The benefits are:
- Numerical Stability: Operations in log-space are generally more numerically stable. Converting the power operation $a^b$ into $e^{b\ln a}$ avoids potential underflow or precision issues that can occur with direct exponentiation `pow()`, especially when gradients are backpropagated. 
    - When training neural networks, gradients are backpropagated. The derivative of $x^n$ involves $x^{n-1}$, which can be volatile. 
    - The exponential function $e^x$ is its own derivative. Working in log-space smooths out the gradients, preventing them from exploding or vanishing as easily as they might with direct high-degree power functions.
- Performance: It transforms a geometric progression (the frequencies) into an arithmetic progression in the exponent. Modern hardware (GPUs) is extremely efficient at computing linear vector arithmetic (multiplication) followed by an element-wise exp, often more so than calculating a generic element-wise `pow` function.

In [ ]:

def sinusoidal_pe(d_model, max_len = 5000, dominator = 10000.0):
    """
    Generate sinusoidal positional encoding.

    Returns:
        pe: Positional encoding tensor of shape (max_len, d_model) 
    """
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(dominator)) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0).transpose(0, 1)
    
    return pe

**2D Positional Encoding**

For images (or any grid-structured input) a token is indexed by a row $h$ and a column $w$ rather than a single position. The standard trick is to split the channel dimension in half: encode the row index with a 1D sinusoidal PE over the first $d_{\text{model}}/2$ channels, and the column index over the remaining $d_{\text{model}}/2$ channels, then concatenate.

$$
PE_{(h, w)} = \big[\; PE^{\text{1D}}_{d/2}(h) \;\;\Vert\;\; PE^{\text{1D}}_{d/2}(w) \;\big]
$$

Each half uses the same $\sin/\cos$ frequencies as the 1D case (with $d \to d/2$). This keeps the encoding additive with the patch embeddings and lets the model recover both coordinates from the channel layout. `d_model` must be divisible by 4 so each half has an even number of channels for the interleaved $\sin/\cos$ pairs.

In [ ]:
def sinusoidal_pe_2d(d_model, height, width, dominator = 10000.0):
    """
    Generate 2D sinusoidal positional encoding.

    Half of the channels encode the row index, the other half the column index.

    Returns:
        pe: Positional encoding tensor of shape (height, width, d_model)
    """
    assert d_model % 4 == 0, "d_model must be divisible by 4 for 2D positional encoding"

    half_dim = d_model // 2  # channels per axis; quarter_dim = half_dim // 2 sin/cos pairs each

    # frequencies shared by both axes, computed once for half_dim channels
    div_term = torch.exp(torch.arange(0, half_dim, 2).float() * (-torch.log(torch.tensor(dominator)) / half_dim))  # (half_dim/2,)

    pos_h = torch.arange(0, height, dtype=torch.float).unsqueeze(1)  # (height, 1)
    pos_w = torch.arange(0, width, dtype=torch.float).unsqueeze(1)   # (width, 1)

    pe = torch.zeros(height, width, d_model)  # (height, width, d_model)

    # rows -> first half of channels, broadcast across width
    # pos_h * div_term: (height, 1) * (half_dim/2,) -> (height, half_dim/2)
    # .unsqueeze(1):                                 -> (height, 1, half_dim/2) broadcast over width
    pe[:, :, 0:half_dim:2] = torch.sin(pos_h * div_term).unsqueeze(1)  # (height, 1, half_dim/2) -> (height, width, half_dim/2)
    pe[:, :, 1:half_dim:2] = torch.cos(pos_h * div_term).unsqueeze(1)  # (height, 1, half_dim/2) -> (height, width, half_dim/2)

    # columns -> second half of channels, broadcast across height
    # pos_w * div_term: (width, 1) * (half_dim/2,) -> (width, half_dim/2)
    # .unsqueeze(0):                               -> (1, width, half_dim/2) broadcast over height
    pe[:, :, half_dim::2] = torch.sin(pos_w * div_term).unsqueeze(0)      # (1, width, half_dim/2) -> (height, width, half_dim/2)
    pe[:, :, half_dim + 1::2] = torch.cos(pos_w * div_term).unsqueeze(0)  # (1, width, half_dim/2) -> (height, width, half_dim/2)

    return pe  # (height, width, d_model)

**FNN**

Feedforward neural network

In [ ]:
class FNN(torch.nn.Module):
    def __init__(self, in_features, hidden_features, out_features = None, bias = True):
        super(FNN, self).__init__()
        out_features = out_features or in_features
        self.linear1 = torch.nn.Linear(in_features, hidden_features, bias)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(hidden_features, out_features, bias)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

**Multi-head Attention**
Multihead attention



In [ ]:
class MHA(torch.nn.Module):
    def __init__(self, emb_dim, num_heads, dropout = 0.0):
        super(MHA, self).__init__()
        # make sure emb_dim is divisible by num_heads
        assert emb_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"

        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.query_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.key_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.value_linear = torch.nn.Linear(emb_dim, emb_dim) 
        self.out_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, query, key = None, value = None, mask = None):
        """
        Args:
            query: Tensor of shape (batch_size, seq_len, emb_dim)
            key: Tensor of shape (batch_size, seq_len, emb_dim) (optional)
            value: Tensor of shape (batch_size, seq_len, emb_dim) (optional)
            mask: Tensor of shape (batch_size, seq_len, seq_len) (optional) 
        """
        if key is None:
            key = query
        if value is None:
            value = query

        # project query, key, value
        query = self.query_linear(query)  # (batch_size, seq_len, emb_dim)
        key = self.key_linear(key)        # (batch_size, seq_len, emb_dim)
        value = self.value_linear(value)  # (batch_size, seq_len, emb_dim) 

        # split into heads
        batch_size = query.size(0)
        query = query.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)
        key = key.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)      # (batch_size, num_heads, seq_len, head_dim)
        value = value.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)

        # scaled dot-product attention
        attn_scores = query @ key.transpose(-2, -1) / (self.head_dim ** 0.5)  # (batch_size, num_heads, seq_len, seq_len)  
        
        # apply mask if provided
        if mask is not None:
            # shape assertion
            assert mask.size() == (batch_size, query.size(2), key.size(2)), "Mask shape must be (batch_size, seq_len, seq_len)"
            attn_scores = attn_scores.masked_fill(mask[:, None, :, :] == 0, float('-inf'))

        # softmax to get attention weights
        attn_weights = torch.softmax(attn_scores, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)

        # apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)

        # weighted sum of values
        attn_output = attn_weights @ value  # (batch_size, num_heads, seq_len, head_dim)

        # concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.emb_dim)  # (batch_size, seq_len, emb_dim)
        attn_output = self.out_linear(attn_output)  # (batch_size, seq_len, emb_dim)

        return attn_output

### MHA backward derivation

The derivation below uses cross-attention because it is the general case. Let $X_Q$, $X_K$, and $X_V$ be the module inputs. For self-attention they are the same tensor $X$.

Use the following dimensions:

| Symbol | Meaning |
|---|---|
| $B$ | batch size |
| $N_h$ | number of attention heads |
| $T_q$ | query sequence length |
| $T_k$ | key/value sequence length |
| $E$ | embedding dimension |
| $D_h=E/N_h$ | dimension of each head |

The forward tensor shapes are

| Tensor | Shape |
|---|---|
| $X_Q$ | $(B,T_q,E)$ |
| $X_K,X_V$ | $(B,T_k,E)$ |
| $W_Q,W_K,W_V,W_O$ | $(E,E)$ |
| $b_Q,b_K,b_V,b_O$ | $(E,)$ |
| $Q$ | $(B,N_h,T_q,D_h)$ |
| $K,V$ | $(B,N_h,T_k,D_h)$ |
| $S,A$ | $(B,N_h,T_q,T_k)$ |
| $C$ | $(B,N_h,T_q,D_h)$ |
| $H,O$ | $(B,T_q,E)$ |

For each head, the forward pass is

$$
\begin{aligned}
Q &= X_QW_Q+b_Q, & K &= X_KW_K+b_K, & V &= X_VW_V+b_V, \\
S &= \frac{QK^T}{\sqrt{D_h}}, & A &= \operatorname{softmax}(S), & C &= AV.
\end{aligned}
$$

The head outputs $C$ are concatenated into $H$, followed by the output projection

$$
O=HW_O+b_O.
$$

Suppose the next layer gives us the upstream gradient $G=\frac{\partial L}{\partial O}$. Work backward by reversing the forward operations.

A gradient always has the same shape as the tensor it differentiates. Therefore $G=dO$ has shape $(B,T_q,E)$.

**1. Output projection**

$$
\frac{\partial L}{\partial W_O}=H^TG, \qquad
\frac{\partial L}{\partial b_O}=\sum G, \qquad
\frac{\partial L}{\partial H}=GW_O^T.
$$

Shape analysis:

$$
\begin{aligned}
dW_O &: (E,E), & db_O &: (E,), & dH &: (B,T_q,E).
\end{aligned}
$$

For $dW_O$, the batch and query-position axes are summed. For $db_O$, both axes are summed. Split $dH$ back into heads to obtain $dC$ with shape $(B,N_h,T_q,D_h)$.

**2. Weighted sum $C=AV$**

$$
dA=dCV^T, \qquad dV=A^TdC.
$$

The batched matrix dimensions are

$$
\begin{aligned}
dA &: (T_q,D_h)(D_h,T_k)=(T_q,T_k), \\
dV &: (T_k,T_q)(T_q,D_h)=(T_k,D_h).
\end{aligned}
$$

Restoring batch and head axes, $dA$ has shape $(B,N_h,T_q,T_k)$ and $dV$ has shape $(B,N_h,T_k,D_h)$.

**3. Row-wise softmax $A=\operatorname{softmax}(S)$**

For one row, the softmax Jacobian is $\operatorname{diag}(a)-aa^T$. Applying it without constructing the Jacobian gives

$$
dS=A\odot\left(dA-\sum_j(dA\odot A)_j\right).
$$

Here $A$, $dA$, and $dS$ all have shape $(B,N_h,T_q,T_k)$. The sum is over the final key axis, producing shape $(B,N_h,T_q,1)$ so that it broadcasts across $T_k$. Masked score positions have zero gradient.

**4. Scaled dot product $S=QK^T/\sqrt{D_h}$**

$$
dQ=\frac{dS K}{\sqrt{D_h}}, \qquad
dK=\frac{dS^T Q}{\sqrt{D_h}}.
$$

For each batch element and head,

$$
\begin{aligned}
dQ &: (T_q,T_k)(T_k,D_h)=(T_q,D_h), \\
dK &: (T_k,T_q)(T_q,D_h)=(T_k,D_h).
\end{aligned}
$$

Thus $dQ$ has shape $(B,N_h,T_q,D_h)$ and $dK$ has shape $(B,N_h,T_k,D_h)$.

**5. Input projections**

After merging the heads back into embedding dimension $E$,

$$
\begin{aligned}
dW_Q &= X_Q^T dQ, & db_Q &= \sum dQ, & dX_Q &= dQW_Q^T, \\
dW_K &= X_K^T dK, & db_K &= \sum dK, & dX_K &= dKW_K^T, \\
dW_V &= X_V^T dV, & db_V &= \sum dV, & dX_V &= dVW_V^T.
\end{aligned}
$$

After merging heads, $dQ$, $dK$, and $dV$ have shapes $(B,T_q,E)$, $(B,T_k,E)$, and $(B,T_k,E)$, respectively. The resulting gradient shapes are

| Gradient | Shape | Reduced axes |
|---|---|---|
| $dW_Q,dW_K,dW_V$ | $(E,E)$ | batch and sequence |
| $db_Q,db_K,db_V$ | $(E,)$ | batch and sequence |
| $dX_Q$ | $(B,T_q,E)$ | none |
| $dX_K,dX_V$ | $(B,T_k,E)$ | none |

The parameter-gradient matrix products implicitly sum over batch and sequence positions. In NumPy, this is implemented by `np.einsum("bte,btf->ef", x, dx)`.

For self-attention, $T_q=T_k=T$ and all three projections depend on the same input. Therefore $dX_Q$, $dX_K$, and $dX_V$ each have shape $(B,T,E)$ and can be added:

$$
\boxed{dX=dX_Q+dX_K+dX_V}.
$$

In [ ]:
import numpy as np


class NumpyMHA:
    """Multi-head attention with a manual NumPy backward pass.

    Dropout is intentionally omitted so forward and backward are deterministic.
    """

    def __init__(self, emb_dim, num_heads, seed=0):
        assert emb_dim % num_heads == 0
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        rng = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(emb_dim)
        self.Wq = rng.normal(0, scale, (emb_dim, emb_dim))
        self.Wk = rng.normal(0, scale, (emb_dim, emb_dim))
        self.Wv = rng.normal(0, scale, (emb_dim, emb_dim))
        self.Wo = rng.normal(0, scale, (emb_dim, emb_dim))
        self.bq = np.zeros(emb_dim)
        self.bk = np.zeros(emb_dim)
        self.bv = np.zeros(emb_dim)
        self.bo = np.zeros(emb_dim)
        self.cache = None
        self.grads = {}

    def _split_heads(self, x):
        batch_size, seq_len, _ = x.shape
        x = x.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(0, 2, 1, 3)

    def _merge_heads(self, x):
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch_size, seq_len, self.emb_dim)

    def forward(self, query, key=None, value=None, mask=None):
        key = query if key is None else key
        value = key if value is None else value

        q_flat = query @ self.Wq + self.bq
        k_flat = key @ self.Wk + self.bk
        v_flat = value @ self.Wv + self.bv
        q = self._split_heads(q_flat)
        k = self._split_heads(k_flat)
        v = self._split_heads(v_flat)

        scores = q @ k.transpose(0, 1, 3, 2) / np.sqrt(self.head_dim)
        mask_bool = None
        if mask is not None:
            mask_bool = np.asarray(mask, dtype=bool)
            assert mask_bool.shape == (query.shape[0], query.shape[1], key.shape[1])
            assert np.all(mask_bool.any(axis=-1)), "Every query must attend to at least one key"
            scores = np.where(mask_bool[:, None, :, :], scores, -np.inf)

        scores_shifted = scores - np.max(scores, axis=-1, keepdims=True)
        exp_scores = np.exp(scores_shifted)
        attn = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        context = attn @ v
        merged = self._merge_heads(context)
        output = merged @ self.Wo + self.bo

        self.cache = (query, key, value, q, k, v, attn, merged, mask_bool)
        return output

    def backward(self, doutput):
        """Return separate gradients for the query, key, and value inputs."""
        query, key, value, q, k, v, attn, merged, mask_bool = self.cache

        self.grads["Wo"] = np.einsum("bte,btf->ef", merged, doutput)
        self.grads["bo"] = doutput.sum(axis=(0, 1))
        dmerged = doutput @ self.Wo.T
        dcontext = self._split_heads(dmerged)

        dattn = dcontext @ v.transpose(0, 1, 3, 2)
        dv = attn.transpose(0, 1, 3, 2) @ dcontext

        # Vector-Jacobian product for row-wise softmax.
        dscores = attn * (dattn - np.sum(dattn * attn, axis=-1, keepdims=True))
        if mask_bool is not None:
            dscores = np.where(mask_bool[:, None, :, :], dscores, 0.0)

        scale = 1.0 / np.sqrt(self.head_dim)
        dq = (dscores @ k) * scale
        dk = (dscores.transpose(0, 1, 3, 2) @ q) * scale
        dq_flat = self._merge_heads(dq)
        dk_flat = self._merge_heads(dk)
        dv_flat = self._merge_heads(dv)

        self.grads["Wq"] = np.einsum("bte,btf->ef", query, dq_flat)
        self.grads["Wk"] = np.einsum("bte,btf->ef", key, dk_flat)
        self.grads["Wv"] = np.einsum("bte,btf->ef", value, dv_flat)
        self.grads["bq"] = dq_flat.sum(axis=(0, 1))
        self.grads["bk"] = dk_flat.sum(axis=(0, 1))
        self.grads["bv"] = dv_flat.sum(axis=(0, 1))

        dquery = dq_flat @ self.Wq.T
        dkey = dk_flat @ self.Wk.T
        dvalue = dv_flat @ self.Wv.T
        return dquery, dkey, dvalue


For cross-attention, `backward` returns the three separate input gradients. For self-attention, call `forward(x, x, x, mask)` and add them:

```python
dx_query, dx_key, dx_value = mha.backward(doutput)
dx = dx_query + dx_key + dx_value
```

In [ ]:
# Small finite-difference check for one input and one parameter element.
rng = np.random.default_rng(1)
mha = NumpyMHA(emb_dim=4, num_heads=2, seed=2)
x = rng.normal(size=(2, 3, 4))
dout = rng.normal(size=(2, 3, 4))

mha.forward(x, x, x)
dxq, dxk, dxv = mha.backward(dout)
dx = dxq + dxk + dxv

def scalar_loss():
    return np.sum(mha.forward(x, x, x) * dout)

eps = 1e-5
x_index = (0, 1, 2)
x[x_index] += eps
loss_plus = scalar_loss()
x[x_index] -= 2 * eps
loss_minus = scalar_loss()
x[x_index] += eps
numeric_dx = (loss_plus - loss_minus) / (2 * eps)

mha.forward(x, x, x)
mha.backward(dout)
w_index = (1, 2)
mha.Wq[w_index] += eps
loss_plus = scalar_loss()
mha.Wq[w_index] -= 2 * eps
loss_minus = scalar_loss()
mha.Wq[w_index] += eps
numeric_dw = (loss_plus - loss_minus) / (2 * eps)

print("dx error:", abs(dx[x_index] - numeric_dx))
print("dWq error:", abs(mha.grads["Wq"][w_index] - numeric_dw))


## Basic Blocks

**Dropout**
Dropout are used in several places to prevent overfitting and stablizing training.

1. On input embeddings: dropout is applied to the sum of token embeddings + positional encodings before feeding into the first encoder/decoder layer
2. After attention weights: inside the multi-head attention mechanism, dropout is applied to the softmax of attention scores before multiplying by the values of V
3. After each sublayer's output
- Output of the multi-head attention block
    ```
    x = LayerNorm(x + Dropout(Attention(x)))
    ```
- Output of the feed-forward network
    ```
    x = LayerNorm(x + Dropout(FeedForward(x)))
    ```
**Attentions**
The default multi-head attention has a time complexity of $O(N^2d)$ and memory complexity of $O(N^2)$. 
The learned attention has a low-rank property, which can be exploited to reduce the complexity.

Many methods have been proposed to reduce the complexity of self-attention, including:

- Sparse Attention: Only attend to a subset of the input tokens, reducing the number of computations.
- Kernelized Attention: Use kernel methods to approximate the softmax attention mechanism.
- Linear Attention: Reformulate the attention mechanism to have linear complexity with respect to the sequence length.

**Encoder Block**


In [ ]:
class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(EncoderBlock, self).__init__()
        self.mha = MHA(emb_dim, num_heads, dropout)
        self.ffn = FNN(emb_dim, ffn_hidden_dim)
        self.norm1 = torch.nn.LayerNorm(emb_dim)
        self.norm2 = torch.nn.LayerNorm(emb_dim)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)
    
    def forward(self, x, mask = None):
        # multi-head attention: self-attention
        attn_output = self.mha(x, x, x, mask)  # (batch_size, seq_len, emb_dim)

        # add & norm
        x = self.norm1(x + self.dropout1(attn_output))  # (batch_size, seq_len, emb_dim)
  
        # feed-forward network
        ffn_output = self.ffn(x)  # (batch_size, seq_len, emb_dim)
        # add & norm
        x = self.norm2(x + self.dropout2(ffn_output))  # (batch_size, seq_len, emb_dim)

        return x

**Decoder Block**

In [ ]:
class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(DecoderBlock, self).__init__()
        self.self_mha = MHA(emb_dim, num_heads, dropout)
        self.cross_mha = MHA(emb_dim, num_heads, dropout)
        self.ffn = FNN(emb_dim, ffn_hidden_dim)
        self.norm1 = torch.nn.LayerNorm(emb_dim)
        self.norm2 = torch.nn.LayerNorm(emb_dim)
        self.norm3 = torch.nn.LayerNorm(emb_dim)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)
        self.dropout3 = torch.nn.Dropout(dropout)
    
    def forward(self, x, enc_output, tgt_mask = None, memory_mask = None):
        """Args:
            x: Tensor of shape (batch_size, tgt_seq_len, emb_dim)
            enc_output: Tensor of shape (batch_size, src_seq_len, emb_dim)
            tgt_mask: Tensor of shape (batch_size, tgt_seq_len, tgt_seq_len)
            memory_mask: Tensor of shape (batch_size, tgt_seq_len, src_seq_len)
        """
        # masked multi-head attention: causal masked self-attention
        self_attn_output = self.self_mha(x, x, x, tgt_mask)  # (batch_size, tgt_seq_len, emb_dim)

        # add & norm
        x = self.norm1(x + self.dropout1(self_attn_output))  # (batch_size, tgt_seq_len, emb_dim)

        # multi-head attention: cross-attention
        cross_attn_output = self.cross_mha(x, enc_output, enc_output, memory_mask)  # (batch_size, tgt_seq_len, emb_dim)

        # add & norm
        x = self.norm2(x + self.dropout2(cross_attn_output))  # (batch_size, tgt_seq_len, emb_dim)

        # feed-forward network
        ffn_output = self.ffn(x)  # (batch_size, tgt_seq_len, emb_dim)

        # add & norm
        x = self.norm3(x + self.dropout3(ffn_output))  # (batch_size, tgt_seq_len, emb_dim)

        return x


## Encoder/Decoder

In [ ]:
class Encoder(torch.nn.Module):
    def __init__(self, num_layers, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(Encoder, self).__init__()
        self.layers = torch.nn.ModuleList([EncoderBlock(emb_dim, num_heads, ffn_hidden_dim, dropout) for _ in range(num_layers)])
    
    def forward(self, x, mask = None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

class Decoder(torch.nn.Module):
    def __init__(self, num_layers, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(Decoder, self).__init__()
        self.layers = torch.nn.ModuleList([DecoderBlock(emb_dim, num_heads, ffn_hidden_dim, dropout) for _ in range(num_layers)])
    
    def forward(self, x, enc_output, tgt_mask = None, memory_mask = None):
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, memory_mask)
        return x


class Transformer(torch.nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, num_layers, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(Transformer, self).__init__()
        self.src_embedding = torch.nn.Embedding(src_vocab_size, emb_dim)
        self.tgt_embedding = torch.nn.Embedding(tgt_vocab_size, emb_dim)
        self.positional_encoding = sinusoidal_pe(emb_dim)
        self.encoder = Encoder(num_layers, emb_dim, num_heads, ffn_hidden_dim, dropout)
        self.decoder = Decoder(num_layers, emb_dim, num_heads, ffn_hidden_dim, dropout)
        self.output_linear = torch.nn.Linear(emb_dim, tgt_vocab_size)
    
    def forward(self, src_input_ids, tgt_input_ids, src_mask = None, tgt_mask = None):
        # embed source and target sequences
        src_embedded = self.src_embedding(src_input_ids) + self.positional_encoding[:src_input_ids.size(1), :].to(src_input_ids.device)  # (batch_size, src_seq_len, emb_dim)
        tgt_embedded = self.tgt_embedding(tgt_input_ids) + self.positional_encoding[:tgt_input_ids.size(1), :].to(tgt_input_ids.device)  # (batch_size, tgt_seq_len, emb_dim)

        # encode source sequence
        enc_output = self.encoder(src_embedded, src_mask)  # (batch_size, src_seq_len, emb_dim)

        # decode target sequence
        dec_output = self.decoder(tgt_embedded, enc_output, tgt_mask, src_mask)  # (batch_size, tgt_seq_len, emb_dim)

        # project to vocabulary size
        output_logits = self.output_linear(dec_output)  # (batch_size, tgt_seq_len, tgt_vocab_size)

        return output_logits